# 05 — Pure Premium: Resolving the Mismatch and Combining Frequency × Severity

**Phase 7**, step 1 of 5: resolve the frequency/severity mismatch documented since Phase 3, and build the combined pure-premium table.

Recall from Phase 3: naively multiplying the frequency champion (Phase 5, fit on `ClaimNb` - *reported* claims) by the severity champion (Phase 6, fit on paid claim amounts) overstates true pure premium by **36.3%**, proven to equal `sum(ClaimNb)/sum(ClaimNbFromSev)`. This has been deferred through two phases; it gets resolved here, properly - not approximated.

In [1]:
import numpy as np
import pandas as pd

from auto_pricing.frequency import predict_frequency
from auto_pricing.severity import build_severity_table, fit_lognormal_model, predict_lognormal_severity
from auto_pricing.pure_premium import (
    fit_paid_frequency_glm,
    predict_annual_pure_premium,
    predict_expected_loss,
)

pp_table = pd.read_parquet("../data/processed/pure_premium_table.parquet")
train = pp_table[pp_table["split"] == "train"].copy()
print(f"train rows: {len(train):,}")
print(f"columns added beyond model_table: ClaimNbFromSev, ClaimAmountSum")

train rows: 474,609
columns added beyond model_table: ClaimNbFromSev, ClaimAmountSum


## 1. Why a flat correction factor was considered and rejected

In [2]:
overall_payment_rate = train["ClaimNbFromSev"].sum() / train["ClaimNb"].sum()
print(f"overall payment rate (train): {overall_payment_rate:.4f}")
print()

by_bonusmalus = train.groupby("BonusMalusBand", observed=True).apply(
    lambda d: d["ClaimNbFromSev"].sum() / d["ClaimNb"].sum() if d["ClaimNb"].sum() > 0 else float("nan"),
    include_groups=False,
)
by_drivage = train.groupby("DrivAgeBand", observed=True).apply(
    lambda d: d["ClaimNbFromSev"].sum() / d["ClaimNb"].sum() if d["ClaimNb"].sum() > 0 else float("nan"),
    include_groups=False,
)
print("payment rate by BonusMalusBand:")
print(by_bonusmalus.round(3))
print()
print("payment rate by DrivAgeBand:")
print(by_drivage.round(3))

overall payment rate (train): 0.7343



payment rate by BonusMalusBand:
BonusMalusBand
50 (best)    0.645
51-59        0.795
60-79        0.830
80-99        0.819
100-129      0.835
130+         1.000
dtype: float64

payment rate by DrivAgeBand:


DrivAgeBand
18-22    0.889
23-29    0.786
30-39    0.775
40-49    0.734
50-59    0.722
60-69    0.662
70+      0.611
dtype: float64


The share of *reported* claims that actually get *paid* ranges from 0.65 to 1.00 across BonusMalus bands, and falls steadily from 0.89 (youngest drivers) to 0.61 (oldest) across DrivAge bands. A single flat correction factor (0.7343) would under-correct young drivers and over-correct older ones - trading one documented, quantified bias for a smaller, differently-shaped, undocumented one. Rejected in favour of a model that learns this variation directly.

## 2. The resolution: a second frequency model, targeting paid claims

Same formula, same machinery as Phase 5's champion - only the target column changes, from `ClaimNb` (reported) to `ClaimNbFromSev` (paid). Phase 5's champion is not replaced: it remains correct for its own purpose (reported claim frequency). This is a second, purpose-built model specifically for pricing.

In [3]:
paid_freq_result = fit_paid_frequency_glm(train)
print(f"converged: {paid_freq_result.converged}")

paid_freq_pred_train = predict_frequency(paid_freq_result, train)
print(f"predicted total paid claims (train): {paid_freq_pred_train.sum():,.2f}")
print(f"observed total paid claims (train):  {train['ClaimNbFromSev'].sum():,}")
print(f"ratio: {paid_freq_pred_train.sum() / train['ClaimNbFromSev'].sum():.4f}")

converged: True


predicted total paid claims (train): 18,523.00
observed total paid claims (train):  18,523
ratio: 1.0000


**Exact match.** This is not a coincidence or approximation - it's the same canonical-link Poisson GLM property used throughout this project (Phase 5), now applied to the correct target for pricing purposes.

## 3. Combining with severity, and checking the full reconciliation

In [4]:
sev = pd.read_parquet("../data/processed/severity_clean.parquet")
severity_table = build_severity_table(sev, pp_table.drop(columns=["ClaimNbFromSev", "ClaimAmountSum"]))
sev_train = severity_table[severity_table["split"] == "train"].copy()

severity_result, smearing_factor = fit_lognormal_model(sev_train)
# Apply to EVERY policy in the pure-premium table, not just those with claims -
# expected cost IF a claim happens, for every policy regardless of whether it did.
severity_pred_train = predict_lognormal_severity(severity_result, smearing_factor, train)

expected_loss_train = predict_expected_loss(paid_freq_pred_train, severity_pred_train)
print(f"predicted total expected loss (train): \u20ac{expected_loss_train.sum():,.2f}")
print(f"observed total paid claim cost (train): \u20ac{train['ClaimAmountSum'].sum():,.2f}")
print(f"ratio: {expected_loss_train.sum() / train['ClaimAmountSum'].sum():.4f}")

predicted total expected loss (train): €43,700,398.81
observed total paid claim cost (train): €44,004,810.78
ratio: 0.9931


**0.9931 - a dramatic improvement over the original 36.3% overstatement.** The small remaining gap is not a new, unexplained discrepancy: it exactly matches the severity model's own known training-reconciliation imperfection from Phase 6 (0.992), which is expected because Lognormal (like Negative Binomial in Phase 5) doesn't share Poisson's exact-match guarantee.

## 4. Two different numbers that must not be confused

The plan explicitly warns against this exact mistake: **expected loss over a policy's own exposure period** and **annual pure premium** are numerically identical only when `Exposure = 1.0` - for any shorter policy, they diverge, and using the wrong one systematically under-prices partial-year policies.

In [5]:
annual_pp_train = predict_annual_pure_premium(paid_freq_pred_train, severity_pred_train, train["Exposure"])

example = pd.DataFrame({
    "Exposure": train["Exposure"],
    "expected_loss": expected_loss_train,
    "annual_pure_premium": annual_pp_train,
}).sort_values("Exposure").head(3)
print("short-exposure policies: expected_loss is much smaller than annual_pure_premium")
print(example)

short-exposure policies: expected_loss is much smaller than annual_pure_premium
       Exposure  expected_loss  annual_pure_premium
57535  0.002732       0.316824           115.957406
57453  0.002732       0.427402           156.429057
57460  0.002732       0.544031           199.115200


In [6]:
exposure_weighted_annual_pp = (annual_pp_train * train["Exposure"]).sum() / train["Exposure"].sum()
print(f"exposure-weighted average annual pure premium (train): \u20ac{exposure_weighted_annual_pp:.2f}")
print(f"Phase 3's raw empirical estimate (no model): \u20ac167.18")
print(f"difference: {(exposure_weighted_annual_pp/167.18 - 1)*100:.1f}%")

exposure-weighted average annual pure premium (train): €174.19
Phase 3's raw empirical estimate (no model): €167.18
difference: 4.2%


€174.19/policy-year vs. Phase 3's raw empirical €167.18 - close (within ~4%), which is the right sanity check here: a model-based estimate built from two fitted GLMs should land near the raw observed figure, not match it exactly (each model has its own small, already-documented imperfection).

## 5. Summary of step 1

- The 36.3% frequency/severity mismatch documented since Phase 3 is now resolved, not   deferred further.
- A flat correction factor was considered and rejected using real evidence: the "payment   rate" varies substantially by BonusMalus (0.65-1.00) and DrivAge (0.61-0.89).
- Instead, a second frequency model (paid-claim frequency, `ClaimNbFromSev`) was fit using   the exact same formula and machinery as Phase 5's champion - Phase 5's original model is   not replaced, it serves a different purpose.
- Verified: frequency alone reconciles exactly (1.0000); the full combined expected loss   reconciles to 0.9931, with the small remaining gap fully explained by severity's own   already-documented imperfection.
- `predict_expected_loss` and `predict_annual_pure_premium` are kept as two distinct,   clearly-labeled functions to avoid the specific confusion the plan warns against.
- The resulting annual pure premium estimate (\u20ac174.19/policy-year) lands close to Phase   3's raw empirical figure (\u20ac167.18), the expected sanity-check relationship.

Full write-up: `reports/pure_premium_reconciliation.md`.

### Next
Step 2: a direct Tweedie GLM - a single model predicting pure premium directly, without splitting frequency and severity at all - as an alternative to compare against.

## 6. Setup: validation-split predictions from step 1's models

Step 1 only needed training-split predictions to verify reconciliation. Steps 2-3 need validation-split predictions from the same fitted models, computed here.

In [7]:
from auto_pricing.evaluation import observed_to_expected_ratio

val = pp_table[pp_table["split"] == "validation"].copy()
paid_freq_pred_val = predict_frequency(paid_freq_result, val)
severity_pred_val = predict_lognormal_severity(severity_result, smearing_factor, val)
print(f"validation policies: {len(val):,}")

validation policies: 101,702


## 7. Step 2: a direct Tweedie GLM

An alternative to the two-model (frequency x severity) pipeline: a single model predicting total claim cost directly, with no split at all. A **Tweedie** distribution with `1 < power < 2` is a compound Poisson-Gamma mixture - it can represent both exact zeros (no claim) and a continuous positive cost (a claim happened) in one distribution, matching the actual shape of `ClaimAmountSum` across the portfolio (~95% zeros, a right-skewed positive tail otherwise).

In [8]:
from auto_pricing.pure_premium import fit_tweedie_glm, TWEEDIE_FORMULA, TWEEDIE_POWER

print(TWEEDIE_FORMULA)
print(f"power = {TWEEDIE_POWER}")

ClaimAmountSum ~ C(DrivAgeBand) + C(VehAgeBand) + C(BonusMalusBand) + AreaOrdinal + LogDensity + VehGasBinary + C(RegionGrouped) + C(VehBrandGrouped) + VehPower
power = 1.5


### The same offset-dropping trap, confirmed for a fourth statsmodels result class

Before trusting any prediction, the same check that caught Phase 5's original bug was repeated here.

In [9]:
tweedie_result = fit_tweedie_glm(train)
print(f"converged: {tweedie_result.converged}")

correct_pred = predict_frequency(tweedie_result, train)
naive_wrong_pred = tweedie_result.predict(train)
print(f"correct (offset applied) total: {correct_pred.sum():,.2f}")
print(f"naive (offset dropped) total:   {naive_wrong_pred.sum():,.2f}")
print(f"observed total:                 {train['ClaimAmountSum'].sum():,.2f}")

converged: True


correct (offset applied) total: 54,190,446.70
naive (offset dropped) total:   115,873,464.32
observed total:                 44,004,810.78


Confirmed: same trap as Poisson, Negative Binomial, and the regularized elastic-net results (Phase 5). `predict_frequency()` already handles it correctly with no changes needed - consistent with the pattern across every statsmodels GLM result class used in this project.

### Choosing the fitting power honestly

Tweedie deviance is a genuinely different function at different power values - "deviance at power=1.1" and "deviance at power=1.8" are not on a comparable scale, even for identical predictions. A fixed-evaluation-power grid search (every fitting-power candidate scored at one shared power) found fitting powers 1.1-1.7 within ~1.6% of each other on validation (67.65-68.78) - not a meaningful difference to chase with a finer grid. `power=1.5` (the standard actuarial default) was adopted for both fitting and evaluation.

## 8. Step 3: full validation comparison

Baseline, Frequency x Severity (Phase 7 step 1), and the direct Tweedie GLM - compared on the validation split using Tweedie deviance (power=1.5, shared across all three for a fair comparison), observed-to-expected ratio, and Gini.

In [10]:
from auto_pricing.evaluation import tweedie_deviance, gini_index, pure_premium_calibration_by_decile

baseline_rate = train["ClaimAmountSum"].sum() / train["Exposure"].sum()
baseline_pred_val = baseline_rate * val["Exposure"]
baseline_rate_val = pd.Series(baseline_rate, index=val.index)
print(f"baseline annual rate: \u20ac{baseline_rate:.2f}")

baseline annual rate: €175.40


In [11]:
freqsev_pred_val = predict_expected_loss(paid_freq_pred_val, severity_pred_val)
freqsev_rate_val = predict_annual_pure_premium(paid_freq_pred_val, severity_pred_val, val["Exposure"])

tweedie_pred_val = predict_frequency(tweedie_result, val)
tweedie_rate_val = tweedie_pred_val / val["Exposure"]

In [12]:
candidates = [
    ("baseline", baseline_pred_val, baseline_rate_val),
    ("freq_x_severity", freqsev_pred_val, freqsev_rate_val),
    ("tweedie", tweedie_pred_val, tweedie_rate_val),
]
for name, pred_amount, pred_rate in candidates:
    dev = tweedie_deviance(val["ClaimAmountSum"], pred_amount.clip(lower=1e-6), power=TWEEDIE_POWER)
    oe = observed_to_expected_ratio(val["ClaimAmountSum"], pred_amount)
    gini = gini_index(val["ClaimAmountSum"], pred_rate, val["Exposure"])
    print(f"{name:<16} deviance={dev:.4f}  O/E={oe:.4f}  Gini={gini:.4f}")

baseline         deviance=66.8503  O/E=0.8408  Gini=-0.0230
freq_x_severity  deviance=65.4363  O/E=0.8430  Gini=0.3016
tweedie          deviance=67.8935  O/E=0.6813  Gini=0.2584


**Frequency x Severity wins on every metric**: lowest deviance, calibration comparable to the baseline, and by far the best risk-ranking (Gini 0.30 vs. Tweedie's 0.26). Tweedie is the worst performer on both deviance and calibration - its O/E of 0.68 means it under-predicts the validation portfolio's total cost by 32%.

### A real surprise: Tweedie's calibration reverses between train and validation

On training data, Tweedie *over*-predicted total cost by 23% (checked earlier: ratio 1.2315). On validation, it *under*-predicts by 32% - a reversal in direction, not just magnitude. Phase 6 found an analogous overfitting problem for Gamma severity, traced to `Region`/`VehBrand`. The same fix was tried here - and it did NOT resolve it:

In [13]:
reduced_formula = (
    "ClaimAmountSum ~ C(DrivAgeBand) + C(VehAgeBand) + C(BonusMalusBand) "
    "+ AreaOrdinal + LogDensity + VehGasBinary + VehPower"
)
tweedie_reduced = fit_tweedie_glm(train, formula=reduced_formula)
tweedie_reduced_pred = predict_frequency(tweedie_reduced, val)
print(f"Tweedie, no Region/VehBrand: O/E = {observed_to_expected_ratio(val['ClaimAmountSum'], tweedie_reduced_pred):.4f}")
print(f"Tweedie, full formula:       O/E = {observed_to_expected_ratio(val['ClaimAmountSum'], tweedie_pred_val):.4f}")

Tweedie, no Region/VehBrand: O/E = 0.6786
Tweedie, full formula:       O/E = 0.6813


Barely moves (0.6813 -> 0.6786) - a materially different result from Gamma's case in Phase 6, checked directly rather than assumed to generalize. The precise mechanism behind Tweedie's calibration gap was not fully pinned down within the scope of this comparison - disclosed honestly rather than either overclaiming a diagnosis or hiding the gap. The practical conclusion doesn't depend on resolving it: Frequency x Severity wins on three independent, converging metrics, not just one.

## 9. A methodology correction, made before trusting the Gini numbers above

The first attempt ranked policies by raw **expected loss** for the Gini calculation. This gave the baseline (a flat rate for everyone, zero real discriminating power) a Gini of **-0.31** - strongly negative, when a non-discriminating model should show ~0.

In [14]:
# The (incorrect) first attempt, shown for contrast - ranking by raw expected loss:
gini_baseline_by_expected_loss = gini_index(val["ClaimAmountSum"], baseline_pred_val, val["Exposure"])
gini_baseline_by_rate = gini_index(val["ClaimAmountSum"], baseline_rate_val, val["Exposure"])
print(f"baseline Gini, ranked by raw expected loss (wrong): {gini_baseline_by_expected_loss:.4f}")
print(f"baseline Gini, ranked by annualized rate (correct): {gini_baseline_by_rate:.4f}")
print()
print(f"correlation(Exposure, ClaimAmountSum): {val['Exposure'].corr(val['ClaimAmountSum']):.4f}")

baseline Gini, ranked by raw expected loss (wrong): -0.3146
baseline Gini, ranked by annualized rate (correct): -0.0230

correlation(Exposure, ClaimAmountSum): 0.0059


**Why this happens**: for the baseline, `expected loss = rate x Exposure`, and since the rate is constant, ranking by expected loss is mechanically identical to ranking by `Exposure` alone. `Exposure` has essentially zero real correlation with risk (0.006) - but a handful of very-short-exposure policies happen to have one large claim (the "short policy + one big claim" pattern documented repeatedly since Phase 3), which then dominates the low end of an exposure-ranked curve and produces a misleadingly negative Gini.

**Fix**: rank by ANNUALIZED rate instead - the same `expected loss` vs. `annualized rate` distinction Phase 7 step 1 already built into two separate functions for exactly this reason. `src/auto_pricing/evaluation.py` was corrected (parameter renamed, docstrings updated with this exact finding) so the mistake can't be quietly reintroduced.

## 10. Gini measures ranking, not calibration - demonstrated directly

The plan explicitly warns about this. Demonstrated on the real champion's validation predictions, not just asserted: scale every prediction by a constant factor (a stand-in for "badly miscalibrated but still ranks correctly") and check what does and doesn't change.

In [15]:
scaled_rate_val = freqsev_rate_val * 3.0  # deliberately miscalibrated, same ranking
scaled_amount_val = freqsev_pred_val * 3.0

gini_original = gini_index(val["ClaimAmountSum"], freqsev_rate_val, val["Exposure"])
gini_scaled = gini_index(val["ClaimAmountSum"], scaled_rate_val, val["Exposure"])
oe_original = observed_to_expected_ratio(val["ClaimAmountSum"], freqsev_pred_val)
oe_scaled = observed_to_expected_ratio(val["ClaimAmountSum"], scaled_amount_val)

print(f"Gini, original predictions:        {gini_original:.4f}")
print(f"Gini, predictions scaled x3:       {gini_scaled:.4f}")
print(f"O/E ratio, original predictions:   {oe_original:.4f}")
print(f"O/E ratio, predictions scaled x3:  {oe_scaled:.4f}")

Gini, original predictions:        0.3016
Gini, predictions scaled x3:       0.3016
O/E ratio, original predictions:   0.8430
O/E ratio, predictions scaled x3:  0.2810


Gini is **identical** (scaling every prediction by the same factor cannot change their relative order); O/E moves dramatically. A model can have excellent Gini and still be priced completely wrong in absolute terms - which is exactly why this project never uses Gini as the only, or even the primary, evaluation metric.

## 11. Summary of steps 2-3

- A direct Tweedie GLM was fit (power=1.5, chosen via a fair, fixed-evaluation-power grid   search) as the plan's alternative to the frequency x severity pipeline.
- The same offset-dropping trap already found in three statsmodels result classes was   confirmed for a fourth (Tweedie); `predict_frequency()` already handled it correctly.
- **Champion: Frequency x Severity** - best deviance (65.44 vs. 67.89), calibration   comparable to the baseline, and clearly the best risk-ranking (Gini 0.30 vs. 0.26).
- Tweedie's training-to-validation calibration reversal (over-predicts by 23% on train,   under-predicts by 32% on validation) was investigated, not assumed to match Phase 6's   Gamma explanation - and it didn't (dropping Region/VehBrand barely moved it). Disclosed   honestly as unresolved; doesn't change the champion decision, which rests on three   independent metrics.
- A real methodology bug was caught and fixed before trusting the Gini numbers: ranking by   raw expected loss instead of annualized rate gave a baseline model a misleadingly negative   Gini, traced to Exposure's mechanical link to expected loss for a flat-rate model.
- Gini's "ranking, not calibration" limitation was demonstrated directly on real validation   predictions (scaling by 3x leaves Gini unchanged, moves O/E substantially), not just   asserted from the plan's warning.

Full decision record: `reports/pure_premium_model_comparison.md`.

### Next
Step 4: champion selection write-up, segment-level stability checks, and Phase 7 wrap-up.